# Mechanistic Interpretability Notebook
This notebook explores mechanistic interpretability techniques for transformer models. It includes model loading, activation patching, attention head analysis, feature extraction, causal interventions, and layer-wise representation inspection.


In [ ]:
# Section 1: Import Required Libraries
import os
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.nn import functional as F

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / "mechanistic_interpretability").exists():
    REPO = REPO.parent
if not (REPO / "mechanistic_interpretability").exists():
    raise FileNotFoundError("Could not locate repository root containing mechanistic_interpretability")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
    print(f"Added repo root to PYTHONPATH: {REPO}")

try:
    from captum.attr import LayerIntegratedGradients
    has_captum = True
except ImportError:
    LayerIntegratedGradients = None
    has_captum = False
    print("Captum not installed. Some interpretability utilities will be skipped.")

print("PyTorch version:", torch.__version__)
print("Transformers and utilities imported.")


In [ ]:
# Cell: Clone a remote repo into the notebook environment
# Set `REPO_URL` to the GitHub URL you want to clone.
REPO_URL = "https://github.com/<owner>/<repo>.git"  # <-- EDIT THIS
TARGET_DIR = "cloned_repo"

import os, subprocess
if not os.path.exists(TARGET_DIR):
    if REPO_URL.startswith("https://github.com/<owner>"):
        print("Please set REPO_URL to the repository you want to clone before running this cell.")
    else:
        print(f"Cloning {REPO_URL} into {TARGET_DIR}...")
        subprocess.run(["git", "clone", REPO_URL, TARGET_DIR], check=True)
else:
    print(f"Target directory '{TARGET_DIR}' already exists; skipping clone.")


In [ ]:
# Cell: Install requirements from cloned repo (if present)
import os
import sys
import subprocess
REQ_PATH = os.path.join("cloned_repo", "requirements.txt")
if os.path.exists(REQ_PATH):
    print(f"Installing requirements from {REQ_PATH}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", REQ_PATH], check=True)
else:
    print(f"No requirements.txt found at {REQ_PATH}. You can install repo deps manually.")


In [ ]:
# Cell: W&B login (interactive)
try:
    import wandb
except Exception:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "wandb"], check=True)
    import wandb

from getpass import getpass
api_key = getpass("W&B API key (leave blank to use existing env/auth): ")
if api_key:
    wandb.login(key=api_key)
    print("W&B logged in using provided key.")
else:
    try:
        ok = wandb.login()
        print("W&B login status:", ok)
    except Exception as e:
        print("W&B login failed or skipped:", e)

# Optional: set project and config defaults
wandb_project = "mechanistic-interpretability"
wandb.init(project=wandb_project, reinit=True)
print(f"W&B initialized project={wandb_project}")


In [ ]:
# Cell: Example - run pipeline script from the repository
import os, sys, subprocess
# Edit `SCRIPT` to the pipeline script you want to run (relative to repo root)
SCRIPT = "mechanistic_interpretability/pipeline/extract_activations.py"
if os.path.exists(SCRIPT):
    cmd = [sys.executable, SCRIPT]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print(f"Script not found: {SCRIPT}. Update SCRIPT to point to a valid pipeline script.")


## Section 2: Load and Prepare a Pretrained Model
Load a pretrained transformer and prepare a small dataset for interpretability experiments.


In [ ]:
MODEL_NAME = "distilgpt2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True,
    output_attentions=True,
    dtype=(torch.float16 if torch.cuda.is_available() else torch.float32),
).to(DEVICE)
model.eval()

sample_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "In the future, language models will help scientists discover new ideas.",
    "A causal intervention helps identify which parts of a network matter.",
]

inputs = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
print("Model loaded and sample batch tokenized.")


## Section 3: Implement Activation Patching
Activation patching replaces intermediate activations to test causal effects on model output.


In [ ]:
def make_activation_patch_hook(replacement_tensor):
    def hook(module, input_, output):
        if isinstance(output, tuple):
            return (replacement_tensor,) + output[1:]
        return replacement_tensor
    return hook


def patch_layer_and_run(model, tokenizer, text_batch, layer_index, quant_func):
    inputs = tokenizer(text_batch, return_tensors="pt", truncation=True, padding=True).to(DEVICE)
    with torch.no_grad():
        clean_out = model(**inputs, output_hidden_states=True)
    clean_layer = clean_out.hidden_states[layer_index].detach()
    patched = quant_func(clean_layer)

    handle = model.transformer.h[layer_index - 1].register_forward_hook(
        make_activation_patch_hook(patched)
    )
    with torch.no_grad():
        out = model(**inputs)
    handle.remove()

    return out.logits, clean_layer, patched


def fake_quantize(tensor, bits=8):
    qmin, qmax = -2 ** (bits - 1), 2 ** (bits - 1) - 1
    min_val, max_val = tensor.min(), tensor.max()
    scale = (max_val - min_val).clamp(min=1e-8) / (qmax - qmin)
    zp = (qmin - (min_val / scale).round()).clamp(qmin, qmax)
    q = (tensor / scale + zp).round().clamp(qmin, qmax)
    return (q - zp) * scale

logits, clean_layer, patched_layer = patch_layer_and_run(
    model,
    tokenizer,
    sample_texts,
    layer_index=3,
    quant_func=lambda x: fake_quantize(x, bits=8),
)
print("Activation patching executed on layer 3.")


## Section 4: Analyze Attention Head Patterns
Extract and visualize attention head weights to understand where the model attends.


In [ ]:
with torch.no_grad():
    outputs = model(**inputs)
    attentions = outputs.attentions

print(f"Extracted {len(attentions)} attention layers.")

layer_to_plot = 0
head_to_plot = 0
attn_weights = attentions[layer_to_plot][0, head_to_plot].cpu().numpy()

plt.figure(figsize=(6, 5))
plt.imshow(attn_weights, cmap="viridis")
plt.title(f"Layer {layer_to_plot + 1}, Head {head_to_plot + 1} Attention")
plt.xlabel("Key Position")
plt.ylabel("Query Position")
plt.colorbar(label="Attention weight")
plt.show()


## Section 5: Extract and Visualize Neural Network Features
Extract hidden states from different layers and visualize high-level structure.


In [ ]:
hidden_states = outputs.hidden_states

for idx, hidden in enumerate(hidden_states[:5]):
    print(f"Layer {idx}: shape {hidden.shape}")

layer_idx = 3
layer_acts = hidden_states[layer_idx][0].detach().cpu().numpy()
mean_activation = np.linalg.norm(layer_acts, axis=-1)

plt.figure(figsize=(8, 3))
plt.plot(mean_activation, marker="o")
plt.title(f"Mean activation norm by token position - Layer {layer_idx}")
plt.xlabel("Token position")
plt.ylabel("Activation norm")
plt.grid(True)
plt.show()


## Section 6: Perform Causal Intervention Analysis
Conduct causal interventions by ablating or modifying specific components to determine their contribution to model predictions.


In [ ]:
def ablate_neurons(layer_tensor, dims):
    patched = layer_tensor.clone()
    patched[..., dims] = 0.0
    return patched

logits_baseline, _, _ = patch_layer_and_run(model, tokenizer, sample_texts, layer_index=3, quant_func=lambda x: x)
logits_ablate, _, _ = patch_layer_and_run(model, tokenizer, sample_texts, layer_index=3, quant_func=lambda x: ablate_neurons(x, list(range(10))))

loss_fn = torch.nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else -100)
inputs_eval = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
labels = inputs_eval.input_ids[:, 1:]

logits_base = logits_baseline[:, :-1, :].reshape(-1, logits_baseline.size(-1))
labels_flat = labels.reshape(-1)
loss_base = loss_fn(logits_base, labels_flat).item()

logits_ab = logits_ablate[:, :-1, :].reshape(-1, logits_ablate.size(-1))
loss_ablate = loss_fn(logits_ab, labels_flat).item()

print(f"Baseline loss: {loss_base:.4f}, Ablated loss: {loss_ablate:.4f}")


## Section 7: Examine Layer-wise Representations
Analyze how representations change across layers to understand the hierarchical processing of information.


In [ ]:
layer_norms = [hidden_states[i][0].detach().norm(dim=-1).mean().item() for i in range(len(hidden_states))]

plt.figure(figsize=(8, 4))
plt.plot(layer_norms, marker="o")
plt.title("Mean token representation norm across layers")
plt.xlabel("Layer")
plt.ylabel("Mean norm")
plt.grid(True)
plt.show()

from sklearn.metrics.pairwise import cosine_similarity
rep_matrix = np.stack([hidden_states[i][0].mean(axis=0).detach().cpu().numpy() for i in range(len(hidden_states))])
similarity = cosine_similarity(rep_matrix)

plt.figure(figsize=(6, 5))
plt.imshow(similarity, cmap="coolwarm", vmin=0, vmax=1)
plt.title("Layer-wise representation cosine similarity")
plt.colorbar()
plt.xlabel("Layer")
plt.ylabel("Layer")
plt.show()


## Section 8: Pipeline Extraction and SAE Training
This section implements the actual extraction of layer-3 activations, the no-padding contiguous chunk logic, and the SAE training loop for the required bottleneck sizes.


In [ ]:
from pathlib import Path
import glob

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / "mechanistic_interpretability").exists():
    REPO = REPO.parent
if not (REPO / "mechanistic_interpretability").exists():
    raise FileNotFoundError("Could not locate the repository root containing mechanistic_interpretability")

SAVE_DIR = REPO / "mechanistic_interpretability" / "data" / "samples"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repository root: {REPO}")
print(f"Saving extracted activations to: {SAVE_DIR}")

MODEL_NAME = "distilgpt2"
SEQ_LEN = 128
MAX_TOKENS = 200_000
FLUSH_EVERY = 500_000
TARGET_LAYER = 3

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm.auto import tqdm

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True,
    dtype=(torch.float16 if torch.cuda.is_available() else torch.float32),
).to(DEVICE).eval()

print("Starting streaming extraction...")

try:
    dataset = load_dataset("openwebtext", split="train", streaming=True)
except Exception as e:
    raise RuntimeError("Failed to load the openwebtext dataset. Ensure the dataset name is correct and HF Hub access is available.") from e

activation_buffer = []
token_id_buffer = []
token_buffer = []
total_extracted = 0
file_idx = 0

def flush_buffer(act_buf, tok_buf, idx):
    stacked_acts = torch.cat(act_buf, dim=0)
    stacked_tokens = torch.cat(tok_buf, dim=0)
    path = SAVE_DIR / f"acts_{idx:04d}.pt"
    torch.save({"acts": stacked_acts.half(), "token_ids": stacked_tokens}, path)
    print(f"Flushed {stacked_acts.shape[0]:,} activations and {stacked_tokens.numel():,} token ids to {path}")
    return [], []

with torch.no_grad():
    for sample in tqdm(dataset, desc="Extracting", total=500):
        text = sample.get("text", "")
        if not text:
            continue

        ids = tokenizer(
            text,
            add_special_tokens=False,
            truncation=True,
            max_length=model.config.n_ctx,
        )["input_ids"]
        if not ids:
            continue
        token_buffer.extend(ids)

        while len(token_buffer) >= SEQ_LEN:
            chunk = token_buffer[:SEQ_LEN]
            token_buffer = token_buffer[SEQ_LEN:]
            x = torch.tensor([chunk], dtype=torch.long, device=DEVICE)

            outputs = model(x)
            h = outputs.hidden_states[TARGET_LAYER]
            h = h / (h.norm(dim=-1, keepdim=True) + 1e-8)

            activation_buffer.append(h.view(-1, 768).float().cpu())
            token_id_buffer.append(torch.tensor(chunk, dtype=torch.long))
            total_extracted += SEQ_LEN

            if total_extracted % FLUSH_EVERY == 0:
                activation_buffer, token_id_buffer = flush_buffer(activation_buffer, token_id_buffer, file_idx)
                file_idx += 1

            if total_extracted >= MAX_TOKENS:
                break

        if total_extracted >= MAX_TOKENS:
            break

if activation_buffer:
    flush_buffer(activation_buffer, token_id_buffer, file_idx)

print(f"Extraction complete. Total tokens extracted: {total_extracted:,}")

### Section 8-B: Train SAE with bottleneck m=512
This training cell uses 10% sparsity (k=51), batch size 4096, Adam lr=1e-4, and normalizes the decoder only after optimizer step.


In [ ]:
import sys
from pathlib import Path

if 'REPO' not in globals():
    REPO = Path.cwd().resolve()
    while REPO != REPO.parent and not (REPO / "mechanistic_interpretability").exists():
        REPO = REPO.parent
    if not (REPO / "mechanistic_interpretability").exists():
        raise FileNotFoundError("Could not locate repository root containing mechanistic_interpretability")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
    print(f"Added repo root to PYTHONPATH: {REPO}")

from mechanistic_interpretability.models.sae import TopKSparseAutoencoder
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

BOTTLENECK = 512
K = int(0.10 * BOTTLENECK)
LR = 1e-4
BATCH_SIZE = 4096
TARGET_STEPS = 100_000

sae = TopKSparseAutoencoder(d_model=768, d_sae=BOTTLENECK, k=K).to(DEVICE)
optimizer = optim.Adam(sae.parameters(), lr=LR)

act_files = sorted(glob.glob(str(SAVE_DIR / "acts_*.pt")))
assert act_files, "No activation files found. Run the extraction cell first."

sae.train()
global_step = 0

for epoch in range(999):
    for path in act_files:
        payload = torch.load(path, map_location="cpu")
        if isinstance(payload, dict) and "acts" in payload:
            activations = payload["acts"].float()
        else:
            activations = payload.float()

        ds = TensorDataset(activations)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

        for (x,) in loader:
            x = x.to(DEVICE)
            x_recon, feats, l2_loss = sae(x)

            loss = l2_loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sae.parameters(), 1.0)
            optimizer.step()
            sae.set_decoder_norm_to_unit_norm()

            global_step += 1
            if global_step % 100 == 0:
                print(f"step {global_step} | loss={loss.item():.4f} | l0={(feats>0).float().sum(-1).mean().item():.2f}")

            if global_step >= TARGET_STEPS:
                break
        if global_step >= TARGET_STEPS:
            break
    if global_step >= TARGET_STEPS:
        break

OUT_PATH = REPO / "mechanistic_interpretability" / "outputs" / f"sae_m{BOTTLENECK}_k{K}.pt"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(sae.state_dict(), OUT_PATH)
print(f"Saved SAE to {OUT_PATH}")


## Section 9: Quantization Analysis and Metrics
This section computes replacement-hook perplexity, SDS, CKA, MSE, and UMAP visualizations for quantized layer-3 activations.


In [ ]:
import math
import json
from sklearn.manifold import TSNE
import umap

from mechanistic_interpretability.utils.metrics import compute_sds, linear_cka


def make_replacement_hook(replacement_tensor):
    def hook(module, input_, output):
        if isinstance(output, tuple):
            return (replacement_tensor,) + output[1:]
        return replacement_tensor
    return hook


def compute_perplexity_with_quantised_layer3(model, tokenizer, text_batch, quant_fn, device, target_layer=3):
    import torch.nn.functional as F
    inputs = tokenizer(text_batch, return_tensors="pt", truncation=True, max_length=128, padding=True).to(device)
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        out_clean = model(**inputs, output_hidden_states=True)
    h3_clean = out_clean.hidden_states[target_layer].clone()
    h3_quant = quant_fn(h3_clean)

    handle = model.transformer.h[target_layer - 1].register_forward_hook(
        make_replacement_hook(h3_quant)
    )
    with torch.no_grad():
        out_quant = model(**inputs)
    handle.remove()

    logits = out_quant.logits
    shift_logits = logits[:, :-1, :].reshape(-1, logits.size(-1))
    shift_labels = input_ids[:, 1:].reshape(-1)
    loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=tokenizer.pad_token_id or -100)
    return loss.item(), h3_clean, h3_quant


def quantise(h, bits, mode="per_tensor"):
    if mode == "per_tensor":
        t_min, t_max = h.min(), h.max()
        scale = (t_max - t_min).clamp(min=1e-8) / ((2 ** (bits - 1) - 1) - (-(2 ** (bits - 1))))
        zp = (-(2 ** (bits - 1)) - (t_min / scale).round()).clamp(-(2 ** (bits - 1)), 2 ** (bits - 1) - 1)
    else:
        t_min = h.reshape(-1, h.shape[-1]).min(0).values
        t_max = h.reshape(-1, h.shape[-1]).max(0).values
        t_min = t_min.view(*([1] * (h.dim() - 1)), -1)
        t_max = t_max.view(*([1] * (h.dim() - 1)), -1)
        scale = (t_max - t_min).clamp(min=1e-8) / ((2 ** (bits - 1) - 1) - (-(2 ** (bits - 1))))
        zp = (-(2 ** (bits - 1)) - (t_min / scale).round()).clamp(-(2 ** (bits - 1)), 2 ** (bits - 1) - 1)

    q = (h / scale + zp).round().clamp(-(2 ** (bits - 1)), 2 ** (bits - 1) - 1)
    return (q - zp) * scale


# Collect ~10k activations for geometry and analysis
H_clean_list = []
with torch.no_grad():
    eval_dataset = load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True)
    for sample in tqdm(eval_dataset, desc="Collecting activations", total=300):
        ids = tokenizer(sample["text"], add_special_tokens=False)["input_ids"]
        if len(ids) < 128:
            continue
        chunk = ids[:128]
        x = torch.tensor([chunk], dtype=torch.long, device=DEVICE)
        out = model(x)
        h = out.hidden_states[3]
        h = h / (h.norm(dim=-1, keepdim=True) + 1e-8)
        H_clean_list.append(h.view(-1, 768).cpu())
        if len(H_clean_list) * 128 >= 10000:
            break

H_clean_10k = torch.cat(H_clean_list, dim=0)[:10000]
print("Collected", H_clean_10k.shape)

BITS_LIST = [8, 4, 2]
QUANT_MODES = ["per_tensor", "per_feature"]
results = {}

for bits in BITS_LIST:
    for mode in QUANT_MODES:
        key = f"{bits}bit_{mode}"
        print(f"\n=== {key} ===")

        H_quant_10k = quantise(H_clean_10k, bits, mode)
        mse = ((H_clean_10k - H_quant_10k) ** 2).mean().item()
        sds = compute_sds(H_clean_10k.numpy(), H_quant_10k.numpy(), k=32)
        cka = linear_cka(H_clean_10k[:1000].numpy(), H_quant_10k[:1000].numpy())

        eval_loss = 0.0
        eval_count = 0
        eval_dataset = load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True)
        for sample in tqdm(eval_dataset, desc=f"Eval PPL {key}", total=20):
            loss, _, _ = compute_perplexity_with_quantised_layer3(
                model, tokenizer, [sample["text"]], lambda h: quantise(h, bits, mode), DEVICE
            )
            eval_loss += loss
            eval_count += 1
            if eval_count >= 20:
                break

        ppl = math.exp(eval_loss / eval_count)
        results[key] = {"bits": bits, "mode": mode, "mse": mse, "sds": sds, "cka": cka, "ppl": ppl}
        print(f"{key}: MSE={mse:.5f}, SDS={sds:.4f}, CKA={cka:.4f}, PPL={ppl:.2f}")

        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
        emb_clean = reducer.fit_transform(H_clean_10k[:2000].numpy())
        emb_quant = reducer.transform(H_quant_10k[:2000].numpy())

        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.scatter(emb_clean[:, 0], emb_clean[:, 1], s=2, alpha=0.4, c="steelblue")
        plt.title("FP32")
        plt.subplot(1, 2, 2)
        plt.scatter(emb_quant[:, 0], emb_quant[:, 1], s=2, alpha=0.4, c="crimson")
        plt.title(f"{bits}-bit {mode}")
        plt.suptitle(f"UMAP: {key}")
        plt.tight_layout()
        plt.show()

with open(REPO / "mechanistic_interpretability" / "outputs" / "quant_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Quantization sweep complete.")


## Section 10: Damage Ranking and Ablation Analysis
This section ranks neurons by L2 and KL damage, extracts top-activating tokens, and compares top damaged neurons against PPL-critical neurons.


In [ ]:
def rank_neurons_by_damage(H_clean: torch.Tensor, H_quant: torch.Tensor):
    l2_per_dim = ((H_clean - H_quant) ** 2).mean(0)

    mu_c, sig_c = H_clean.mean(0), H_clean.std(0).clamp(min=1e-8)
    mu_q, sig_q = H_quant.mean(0), H_quant.std(0).clamp(min=1e-8)
    kl_per_dim = (
        (sig_q / sig_c).log()
        + (sig_c ** 2 + (mu_c - mu_q) ** 2) / (2 * sig_q ** 2)
        - 0.5
    )

    l2_rank = torch.argsort(l2_per_dim, descending=True)
    kl_rank = torch.argsort(kl_per_dim, descending=True)
    return {
        "l2_per_dim": l2_per_dim,
        "kl_per_dim": kl_per_dim,
        "l2_rank": l2_rank,
        "kl_rank": kl_rank,
    }

H_q4 = quantise(H_clean_10k, 4, "per_tensor")
damage_4bit = rank_neurons_by_damage(H_clean_10k, H_q4)
top20_l2 = damage_4bit["l2_rank"][:20].tolist()
top20_kl = damage_4bit["kl_rank"][:20].tolist()
print("Top-20 most L2-damaged dims:", top20_l2)
print("Top-20 most KL-damaged dims:", top20_kl)


def spectral_analysis(H_clean, H_quant, k=64, label="4bit"):
    H_c = H_clean - H_clean.mean(0)
    H_q = H_quant - H_quant.mean(0)
    _, S_c, Vc = torch.linalg.svd(H_c, full_matrices=False)
    _, S_q, Vq = torch.linalg.svd(H_q, full_matrices=False)

    plt.figure(figsize=(10, 4))
    plt.plot(S_c[:200].cpu().numpy(), label="FP32", lw=2)
    plt.plot(S_q[:200].cpu().numpy(), label=f"{label}", lw=2, linestyle="--")
    plt.legend()
    plt.title(f"Singular Value Spectrum: FP32 vs {label}")
    plt.xlabel("Index")
    plt.ylabel("Singular value")
    plt.show()

    M = Vc[:k] @ Vq[:k].T
    sv = torch.linalg.svdvals(M).clamp(-1, 1)
    angles_deg = torch.acos(sv) * 180 / torch.pi
    print(f"[{label}] SDS={compute_sds(H_clean.numpy(), H_quant.numpy(), k=k):.4f} mean_angle={angles_deg.mean().item():.2f}°")

    plt.figure(figsize=(8, 4))
    plt.plot(angles_deg.cpu().numpy())
    plt.title(f"Principal Angles: FP32 vs {label}")
    plt.xlabel("Principal component index")
    plt.ylabel("Angle (degrees)")
    plt.show()
    return angles_deg

angles = spectral_analysis(H_clean_10k, H_q4, k=64, label="4bit_per_tensor")


def top_activating_tokens_for_dims(dims, n_top=10, max_files=3):
    records = {}
    files = sorted(glob.glob(str(SAVE_DIR / "acts_*.pt")))[:max_files]
    for dim in dims:
        records[dim] = []
    for path in files:
        payload = torch.load(path, map_location="cpu")
        acts = payload["acts"].float()
        token_ids = payload["token_ids"]
        for dim in dims:
            values = acts[:, dim]
            topk = values.topk(n_top)
            for score, idx in zip(topk.values.tolist(), topk.indices.tolist()):
                token_id = int(token_ids[idx].item())
                records[dim].append((score, tokenizer.convert_ids_to_tokens([token_id])[0]))
    for dim in records:
        records[dim] = sorted(records[dim], key=lambda x: x[0], reverse=True)[:n_top]
    return records

records = top_activating_tokens_for_dims(top20_l2[:5], n_top=5)
print("Top activating tokens for top 5 L2-damaged dims:")
for dim, tokens in records.items():
    print(dim, tokens)


def ablation_ppl(model, tokenizer, text_batch, dims_to_zero, device):
    def zero_hook(module, input_, output):
        h = output[0].clone() if isinstance(output, tuple) else output.clone()
        h[:, :, dims_to_zero] = 0.0
        return (h,) + output[1:] if isinstance(output, tuple) else h

    handle = model.transformer.h[TARGET_LAYER - 1].register_forward_hook(zero_hook)
    inputs = tokenizer(text_batch, return_tensors="pt", truncation=True, max_length=128, padding=True).to(device)
    with torch.no_grad():
        out = model(**inputs)
    handle.remove()

    ids = inputs["input_ids"]
    logits = out.logits
    loss = torch.nn.functional.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)), ids[:, 1:].reshape(-1), ignore_index=tokenizer.pad_token_id or -100)
    return math.exp(loss.item())

TOP20 = top20_l2[:20]
RAND20 = torch.randperm(768)[:20].tolist()
eval_texts = [s["text"] for _, s in zip(range(10), load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True))]

ppl_baseline = ablation_ppl(model, tokenizer, eval_texts[:5], [], DEVICE)
ppl_top20 = ablation_ppl(model, tokenizer, eval_texts[:5], TOP20, DEVICE)
ppl_rand20 = ablation_ppl(model, tokenizer, eval_texts[:5], RAND20, DEVICE)

print(f"PPL baseline: {ppl_baseline:.2f}")
print(f"PPL top-20 ablated: {ppl_top20:.2f}  Δ={ppl_top20-ppl_baseline:.2f}")
print(f"PPL rand-20 ablated:{ppl_rand20:.2f}  Δ={ppl_rand20-ppl_baseline:.2f}")


def per_dim_ablation_ppl_delta(model, tokenizer, eval_texts, n_dims=50, device=DEVICE):
    baseline = ablation_ppl(model, tokenizer, eval_texts, [], device)
    deltas = {}
    for dim in range(n_dims):
        deltas[dim] = ablation_ppl(model, tokenizer, eval_texts, [dim], device) - baseline
    return deltas

ppl_deltas = per_dim_ablation_ppl_delta(model, tokenizer, eval_texts[:5], n_dims=50, device=DEVICE)
ppl_rank = sorted(ppl_deltas, key=ppl_deltas.get, reverse=True)
overlap = len(set(TOP20) & set(ppl_rank[:20]))
print(f"Overlap between top-20 L2-damaged and top-20 PPL-critical: {overlap}/20")


## Section 11: Subspace-Preserving Quantisation (SPQ)
Compare a hybrid protected subspace quantization strategy against standard 4-bit quantization.


In [ ]:
class SubspacePreservingQuantiser:
    def __init__(self, H_clean: torch.Tensor, k: int = 32):
        H_c = (H_clean - H_clean.mean(0)).float()
        _, _, V = torch.linalg.svd(H_c, full_matrices=False)
        self.V_k = V[:k].T
        self.mean = H_clean.mean(0)
        self.k = k

    def _fake_quant(self, x: torch.Tensor, bits: int) -> torch.Tensor:
        q_min, q_max = -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
        t_min, t_max = x.min(), x.max()
        scale = (t_max - t_min).clamp(min=1e-8) / (q_max - q_min)
        zp = (q_min - (t_min / scale).round()).clamp(q_min, q_max)
        q = (x / scale + zp).round().clamp(q_min, q_max)
        return (q - zp) * scale

    def quantise(self, H: torch.Tensor, bits_important: int = 8, bits_residual: int = 2) -> torch.Tensor:
        H_c = (H - self.mean).float()
        V_k = self.V_k.to(H.device)
        coords = H_c @ V_k
        h_imp = coords @ V_k.T
        h_res = H_c - h_imp
        h_imp_q = self._fake_quant(h_imp, bits_important)
        h_res_q = self._fake_quant(h_res, bits_residual)
        return (h_imp_q + h_res_q + self.mean.to(H.device)).to(H.dtype)

spq = SubspacePreservingQuantiser(H_clean_10k[:5000], k=32)
H_spq = spq.quantise(H_clean_10k, bits_important=8, bits_residual=2)
H_std4 = quantise(H_clean_10k, 4, "per_tensor")

mse_spq = ((H_clean_10k - H_spq) ** 2).mean().item()
mse_std4 = ((H_clean_10k - H_std4) ** 2).mean().item()
sds_spq = compute_sds(H_clean_10k.numpy(), H_spq.numpy(), k=32)
sds_std4 = compute_sds(H_clean_10k.numpy(), H_std4.numpy(), k=32)
cka_spq = linear_cka(H_clean_10k[:1000].numpy(), H_spq[:1000].numpy())
cka_std4 = linear_cka(H_clean_10k[:1000].numpy(), H_std4[:1000].numpy())

print(f"{'Method':25s} {'CKA':>8s} {'MSE':>12s} {'SDS':>8s}")
print(f"{'Standard 4-bit':25s} {cka_std4:8.4f} {mse_std4:12.5f} {sds_std4:8.4f}")
print(f"{'SPQ (8-bit imp, 2-bit res)':25s} {cka_spq:8.4f} {mse_spq:12.5f} {sds_spq:8.4f}")


### Section 8-C: Train SAE with bottleneck m=1024
Repeat the SAE training with the larger bottleneck and correct sparsity k=102.


In [ ]:
BOTTLENECK = 1024
K = int(0.10 * BOTTLENECK)
LR = 1e-4
BATCH_SIZE = 4096
TARGET_STEPS = 100_000

sae = TopKSparseAutoencoder(d_model=768, d_sae=BOTTLENECK, k=K).to(DEVICE)
optimizer = optim.Adam(sae.parameters(), lr=LR)

act_files = sorted(glob.glob(str(SAVE_DIR / "acts_*.pt")))
assert act_files, "No activation files found. Run the extraction cell first."

sae.train()
global_step = 0

for epoch in range(999):
    for path in act_files:
        payload = torch.load(path, map_location="cpu")
        if isinstance(payload, dict) and "acts" in payload:
            activations = payload["acts"].float()
        else:
            activations = payload.float()

        ds = TensorDataset(activations)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

        for (x,) in loader:
            x = x.to(DEVICE)
            x_recon, feats, l2_loss = sae(x)
            loss = l2_loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sae.parameters(), 1.0)
            optimizer.step()
            sae.set_decoder_norm_to_unit_norm()

            global_step += 1
            if global_step % 100 == 0:
                print(f"step {global_step} | loss={loss.item():.4f} | l0={(feats>0).float().sum(-1).mean().item():.2f}")
            if global_step >= TARGET_STEPS:
                break
        if global_step >= TARGET_STEPS:
            break
    if global_step >= TARGET_STEPS:
        break

OUT_PATH = REPO / "mechanistic_interpretability" / "outputs" / f"sae_m{BOTTLENECK}_k{K}.pt"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(sae.state_dict(), OUT_PATH)
print(f"Saved SAE to {OUT_PATH}")
